# Fixed-Dense-SW: E10 τ probe (CPU only)
Attach **one** completed seed-3 U/R/SW/RG output containing `e2e_pairwise_pilot_v2`. This notebook does not train, use test, or choose κ automatically. It writes a small probe output for the later T4×2 gate.

In [ ]:
import os,subprocess,sys,tempfile,importlib,json
from pathlib import Path
from IPython.display import display
from kaggle_secrets import UserSecretsClient
PROJECT_ROOT=Path('/kaggle/working/new-pruning')
token=UserSecretsClient().get_secret('github_token'); assert token,'Missing Kaggle secret github_token'
with tempfile.TemporaryDirectory() as temporary:
    askpass=Path(temporary)/'askpass.py'
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env=os.environ.copy();env.update(GIT_ASKPASS=str(askpass),GIT_TERMINAL_PROMPT='0',GITHUB_TOKEN_RUNTIME=token)
    subprocess.run(['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)],env=env,check=True)
token=None;os.chdir(PROJECT_ROOT);sys.path.insert(0,str(PROJECT_ROOT))
import rq2_e2e_pairwise_pilot as pilot
import rq2_fixed_dense_sw_gate as gate
assert hasattr(gate,'run_tau_probe') and hasattr(gate,'freeze_policy'), 'Pull the new fixed-dense-SW source revision'
print('Code commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())


In [ ]:
INPUT_ROOT=Path('/kaggle/input')
ROOT=gate.find_completed_seed3_root(INPUT_ROOT)
if ROOT is None: ROOT=gate.materialize_completed_seed3_root(INPUT_ROOT,'/kaggle/working/materialized-fixedsw-source')  # ZIP fallback.
assert (ROOT/'pure_sw/sw_policies/epoch_010.npz').is_file()
print('E10 source:',ROOT)


In [ ]:
OUTPUT_DIR=Path('/kaggle/working/fixed_dense_sw_probe')
table=gate.run_tau_probe(ROOT,OUTPUT_DIR)
display(table[['kappa','tau','entropy','effective_support','support_size','l1_to_uniform','max_probability','marginal_error']])
assert (table.marginal_error<1e-6).all()
print('Read the three regimes, then choose one κ *before* any E2E accuracy. Attach this probe output to the GPU notebook.')
print('Files:',list(OUTPUT_DIR.iterdir()))
